In [1]:
from matplotlib import pyplot as plt
from itertools import product
import seaborn as sns
import numpy as np
import pandas as pd
import os.path as op
import argparse
import re
import glob
# EEG utilities
import mne
from pycrostates.cluster import ModKMeans
from pycrostates.io import read_cluster, ChData
# BIDS utilities
from util.io.bids import DataSink
from bids import BIDSLayout

In [2]:
# constants
BIDS_ROOT = '../data/bids'
DERIV_ROOT = op.join(BIDS_ROOT, 'derivatives')
TASK = 'pitch'
MIN_TRIAL_CUTOFF = 100 # subjects must have this many trials in *every* condition to be included

layout = BIDSLayout(BIDS_ROOT, derivatives = True)

/project/hcn1/.conda/envs/mne/lib/python3.11/site-packages/bids/layout/validation.py:124: UserWarning: The PipelineDescription field was superseded by GeneratedBy in BIDS 1.4.0. You can use ``pybids upgrade`` to update your derivative dataset.
  warnings.warn("The PipelineDescription field was superseded "


In [3]:
def read_epochs(sub, desc):
    '''
    reads and concatenates epochs across runs
    '''
    layout = BIDSLayout(BIDS_ROOT, derivatives = True)
    run = lambda f: int(re.findall('run-(\w+)_', f)[0])
    fnames = layout.get(
        return_type = 'filename',
        subject = sub, desc = desc
        )
    fnames.sort(key = run)
    epochs_all = [mne.read_epochs(f) for f in fnames]
    epochs = mne.concatenate_epochs(epochs_all)
    epochs = epochs.pick('eeg')
    return epochs

In [4]:
subs = layout.get_subjects(scope = 'microstates')
subs = [10, 11, 28, 31, 42]
subs.sort(key = int)

for sub in subs:
    print(f'sub: {sub}')
    
    # get last microstate observed before each stimulus is delivered
    microstate_epochs_1 = read_epochs(sub, 'forMicrostate')
    microstate_epochs_2 = read_epochs(sub, 'forMicrostate2')
    
    # read FFR epochs and assign microstate labels
    ffr_epochs_1 = read_epochs(sub, 'forFFR')
    ffr_epochs_2 = read_epochs(sub, 'forFFR2')

    print(microstate_epochs_1)
    print(microstate_epochs_2)
    print(ffr_epochs_1)
    print(ffr_epochs_2)
    # break

sub: 10
Reading /project/hcn1/Letty/pitch_tracking_attention/analysis/../data/bids/derivatives/microstates/sub-10/sub-10_task-pitch_run-1_desc-forMicrostate_epo.fif.gz ...
    Found the data of interest:
        t =    -200.00 ...       0.00 ms
        0 CTF compensation matrices available
Not setting metadata
3348 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
3348 matching events found
No baseline correction applied


/project/hcn1/.conda/envs/mne/lib/python3.11/site-packages/bids/layout/validation.py:124: UserWarning: The PipelineDescription field was superseded by GeneratedBy in BIDS 1.4.0. You can use ``pybids upgrade`` to update your derivative dataset.
  warnings.warn("The PipelineDescription field was superseded "


Reading /project/hcn1/Letty/pitch_tracking_attention/analysis/../data/bids/derivatives/microstates/sub-10/sub-10_task-pitch_run-1_desc-forMicrostate2_epo.fif.gz ...
    Found the data of interest:
        t =    -250.00 ...       0.00 ms
        0 CTF compensation matrices available
Not setting metadata
3360 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
3360 matching events found
No baseline correction applied


/project/hcn1/.conda/envs/mne/lib/python3.11/site-packages/bids/layout/validation.py:124: UserWarning: The PipelineDescription field was superseded by GeneratedBy in BIDS 1.4.0. You can use ``pybids upgrade`` to update your derivative dataset.
  warnings.warn("The PipelineDescription field was superseded "


Reading /project/hcn1/Letty/pitch_tracking_attention/analysis/../data/bids/derivatives/preprocess_ffr/sub-10/sub-10_task-pitch_run-1_desc-forFFR_epo.fif.gz ...
    Found the data of interest:
        t =    -200.00 ...     400.00 ms
        0 CTF compensation matrices available
Not setting metadata
3348 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
3348 matching events found
Applying baseline correction (mode: mean)


...

Reading /project/hcn1/Letty/pitch_tracking_attention/analysis/../data/bids/derivatives/microstates/sub-42/sub-42_task-pitch_run-1_desc-forMicrostate2_epo.fif.gz ...
    Found the data of interest:
        t =    -250.00 ...       0.00 ms
        0 CTF compensation matrices available
Not setting metadata
3274 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
3274 matching events found
No baseline correction applied


/project/hcn1/.conda/envs/mne/lib/python3.11/site-packages/bids/layout/validation.py:124: UserWarning: The PipelineDescription field was superseded by GeneratedBy in BIDS 1.4.0. You can use ``pybids upgrade`` to update your derivative dataset.
  warnings.warn("The PipelineDescription field was superseded "


Reading /project/hcn1/Letty/pitch_tracking_attention/analysis/../data/bids/derivatives/preprocess_ffr/sub-42/sub-42_task-pitch_run-1_desc-forFFR_epo.fif.gz ...
    Found the data of interest:
        t =    -200.00 ...     400.00 ms
        0 CTF compensation matrices available
Not setting metadata
3180 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
3180 matching events found
Applying baseline correction (mode: mean)


/project/hcn1/.conda/envs/mne/lib/python3.11/site-packages/bids/layout/validation.py:124: UserWarning: The PipelineDescription field was superseded by GeneratedBy in BIDS 1.4.0. You can use ``pybids upgrade`` to update your derivative dataset.
  warnings.warn("The PipelineDescription field was superseded "


Reading /project/hcn1/Letty/pitch_tracking_attention/analysis/../data/bids/derivatives/preprocess_ffr/sub-42/sub-42_task-pitch_run-1_desc-forFFR2_epo.fif.gz ...
    Found the data of interest:
        t =    -250.00 ...     250.00 ms
        0 CTF compensation matrices available
Not setting metadata
3274 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
3274 matching events found
Applying baseline correction (mode: mean)
<EpochsArray |  3180 events (all good), -0.2 – 0 s, baseline off, ~362.6 MB, data loaded,
 '11': 332
 '12': 319
 '13': 344
 '21': 328
 '22': 321
 '23': 332
 '31': 394
 '32': 419
 '33': 391>
<EpochsArray |  3274 events (all good), -0.25 – 0 s, baseline off, ~466.2 MB, data loaded,
 '11': 338
 '12': 334
 '13': 354
 '21': 340
 '22': 331
 '23': 340
 '31': 408
 '32': 433
 '33': 396>
<EpochsArray |  3180 events (all good), -0.2 – 0.4 s, baseline -0.2 – 0 s, ~17.5 MB, data loaded,
 '11': 332
 '12': 319
 '13': 344
 '21': 328